<a href="https://colab.research.google.com/github/FaizaTasneeem/apachejit-defect-prediction/blob/main/apachejit_defect_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install catboost shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 35.8 MB/s  0:00:02


In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/hosseinkshvrz/apachejit/master/dataset/apachejit_total.csv"
df = pd.read_csv(url)
buggy = df[df['buggy'] == True]

print(f"Columns: {', ' .join(df.columns)}")
print(f"Row count: {len(df)}")
print(f"Share of buggy commits: {round((len(buggy)/len(df)*100), 2)} %")


Columns: commit_id, project, buggy, fix, year, author_date, la, ld, nf, nd, ns, ent, ndev, age, nuc, aexp, arexp, asexp
Row count: 106674
Share of buggy commits: 26.47 %


In [ ]:
print(df.dtypes)

commit_id       object
project         object
buggy             bool
fix               bool
year             int64
author_date      int64
la               int64
ld               int64
nf               int64
nd               int64
ns               int64
ent            float64
ndev           float64
age            float64
nuc            float64
aexp             int64
arexp          float64
asexp          float64
dtype: object


In [3]:
df["author_date"] = pd.to_datetime(df["author_date"], unit="s")
print("Earliest commit:", df["author_date"].min())
print("Latest commit:", df["author_date"].max())

Earliest commit: 2003-09-11 14:54:46
Latest commit: 2019-12-26 17:39:53


In [8]:
print("Checking if author_date is sorted in ascending order: ", df["author_date"].is_monotonic_increasing)
print("Checking if author_date is sorted in descending order: ", df["author_date"].is_monotonic_decreasing)

Checking if author_date is sorted in ascending order:  False
Checking if author_date is sorted in descending order:  False


In [ ]:
print(df.isnull().sum())
print("Total missing:", df.isnull().sum().sum())

commit_id      0
project        0
buggy          0
fix            0
year           0
author_date    0
la             0
ld             0
nf             0
nd             0
ns             0
ent            0
ndev           0
age            0
nuc            0
aexp           0
arexp          0
asexp          0
dtype: int64
Total missing: 0


In [ ]:
print("Duplicate commit_ids:", df["commit_id"].duplicated().sum())

Duplicate commit_ids: 0


In [11]:
print("Number of project values:", df["project"].nunique())
print("Number of commits per project:", df["project"].value_counts())

Number of project values: 15
Number of commits per project: project
apache/camel               22700
apache/ignite              12036
apache/hadoop              11964
apache/flink               11691
apache/hbase                8730
apache/cassandra            8159
apache/groovy               8059
apache/hive                 6842
apache/activemq             6126
apache/hadoop-hdfs          2907
apache/kafka                2384
apache/spark                1465
apache/zeppelin             1451
apache/hadoop-mapreduce     1321
apache/zookeeper             839
Name: count, dtype: int64


In [ ]:
from google.colab import data_table

start = 0
data_table.DataTable(df.iloc[start:start + 20000], num_rows_per_page=50)

,commit_id,project,buggy,fix,year,author_date,la,ld,nf,nd,ns,ent,ndev,age,nuc,aexp,arexp,asexp
0,7b8480744ea6e6fb41efd4329bb470c8f3c763db,apache/groovy,False,False,2003,1070355653,372,23,8,3,3,2.669743,0.250000,3.625000,2.125000,243,243.00,0.683585
1,192b631e7be302ecde822546ba70a9853ddbda01,apache/groovy,False,False,2003,1063298262,2,2,2,2,1,1.000000,1.000000,0.000000,2.500000,19,19.00,14.000000
2,0ab6465a7dece117c61c3efd3ec95d20524bdad6,apache/groovy,False,False,2003,1069704572,41,26,3,3,2,1.237612,0.666667,5.333333,45.000000,233,233.00,0.000606
3,449241d5fa1aeadd5eb9fa1280d4104dc0dfb891,apache/groovy,False,False,2003,1068487625,8,6,2,1,1,0.591673,1.000000,3.000000,35.500000,64,64.00,55.000000
4,698df9f5b0816e05bbf97a390428f8ef94e0cdbf,apache/groovy,False,False,2003,1063313516,70,4,6,3,1,2.519672,0.333333,0.000000,6.000000,27,27.00,22.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,8f042fa632bf5617ebc1e22174ccb6be5136119d,apache/flink,False,False,2011,1308755049,231,27,6,1,1,2.576641,0.166667,0.166667,1.500000,334,334.00,34.000000
19996,7569cea558f3804045d28bd73fb8b316cf8affc7,apache/hbase,False,False,2011,1324532669,2,58,3,2,2,1.136477,5.333333,0.000000,966.333333,2615,1425.20,1.966797
19997,883eed0e385e9695a3746f1dede0a15c9fc24427,apache/activemq,False,False,2011,1300921098,181,157,4,1,1,1.724424,0.500000,39.250000,7.750000,52,44.75,40.000000
19998,a60154826eee2f7a77044024c40801ba00156048,apache/flink,False,False,2011,1305889916,86,3,2,1,1,0.264517,0.500000,0.000000,4.500000,107,107.00,65.000000
